# Feature Store

Engenharia de features, codificação, normalização e divisão estratificada para o modelo de churn.

## 1. Importações e carregamento

In [1]:
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

df = pd.read_parquet(Path('../data/interm/base_consolidada.parquet'))
print(f'Base carregada: {df.shape[0]:,} linhas x {df.shape[1]} colunas')

Base carregada: 7,043 linhas x 21 colunas


## 2. Engenharia de features

São criadas variáveis relacionadas a tempo de relacionamento, cobrança, serviços e risco de churn.

In [2]:
# Features de tenure
df['tenure_grupo'] = pd.cut(
    df['tenure'], bins=[0, 12, 24, 48, 72],
    labels=['0-12m', '13-24m', '25-48m', '49-72m'], include_lowest=True
)
df['is_novo_cliente'] = (df['tenure'] <= 12).astype(int)

# Features financeiras
df['charges_por_mes'] = np.where(
    df['tenure'] > 0, df['TotalCharges'] / df['tenure'], df['MonthlyCharges']
)
media_por_contrato = df.groupby('Contract')['MonthlyCharges'].transform('mean')
df['charges_desvio_contrato'] = df['MonthlyCharges'] - media_por_contrato

# Features de serviços
servicos_adicionais = [
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies',
]
df['qtd_servicos'] = df[servicos_adicionais].apply(
    lambda row: (row == 'Yes').sum(), axis=1
)
df['sem_servicos_adicionais'] = (df['qtd_servicos'] == 0).astype(int)

### 2.1 Suporte técnico e score de risco

O score combina características associadas a maior propensão de churn.

In [3]:
# Clientes sem suporte técnico e com fibra têm risco elevado.
df['is_sem_suporte'] = (df['TechSupport'] == 'No').astype(int)
df['fibra_sem_suporte'] = (
    (df['InternetService'] == 'Fiber optic') & (df['TechSupport'] == 'No')
).astype(int)
df['is_sem_seguranca_online'] = (df['OnlineSecurity'] == 'No').astype(int)
df['qtd_servicos_protecao'] = df[
    ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection']
].apply(lambda row: (row == 'Yes').sum(), axis=1)
df['sem_protecao'] = (df['qtd_servicos_protecao'] == 0).astype(int)

# Flags de risco (usadas para scores e interações; algumas serão excluídas após OHE)
df['is_contrato_mensal'] = (df['Contract'] == 'Month-to-month').astype(int)
df['is_fibra_optica'] = (df['InternetService'] == 'Fiber optic').astype(int)
df['is_boleto_eletronico'] = (df['PaymentMethod'] == 'Electronic check').astype(int)
df['is_senior'] = df['SeniorCitizen'].astype(int)

df['score_risco_base'] = (
    df['is_contrato_mensal']
    + df['is_fibra_optica']
    + df['is_boleto_eletronico']
    + df['is_senior']
)
df['score_risco_estendido'] = (
    df['score_risco_base']
    + df['is_sem_suporte']
    + df['is_sem_seguranca_online']
)

# Interações (insights do EDA — contrato mensal × perfil amplifica churn)
df['contrato_mensal_novo'] = df['is_contrato_mensal'] * df['is_novo_cliente']
df['contrato_mensal_fibra'] = df['is_contrato_mensal'] * df['is_fibra_optica']
df['tenure_x_contrato_mensal'] = df['tenure'] * df['is_contrato_mensal']
df['charges_x_contrato_mensal'] = df['MonthlyCharges'] * df['is_contrato_mensal']
df['boleto_x_contrato_mensal'] = df['is_boleto_eletronico'] * df['is_contrato_mensal']
df['fibra_x_sem_protecao'] = df['is_fibra_optica'] * df['sem_protecao']
df['tenure_curto_fibra'] = (
    (df['tenure'] <= 12) & (df['InternetService'] == 'Fiber optic')
).astype(int)
df['alta_cobranca_relativa'] = (df['charges_desvio_contrato'] > 0).astype(int)

print('\nDistribuição do Score de Risco Estendido:')
display(
    df.groupby('score_risco_estendido')['Churn'].agg(['count', 'mean'])
    .rename(columns={'count': 'clientes', 'mean': 'taxa_churn'})
    .assign(taxa_churn=lambda x: (x['taxa_churn'] * 100).round(1))
)


Distribuição do Score de Risco Estendido:


,clientes,taxa_churn
score_risco_estendido,,
0,502,2.8
1,720,7.8
2,1781,8.1
3,1579,22.5
4,1167,43.4
5,950,58.9
6,344,67.4


## 3. Preservação dos valores originais

Valores não normalizados são retidos para interpretação posterior.

In [4]:
df['MonthlyCharges_orig'] = df['MonthlyCharges']
df['TotalCharges_orig'] = df['TotalCharges']
df['tenure_orig'] = df['tenure']

## 4. Encoding das variáveis categóricas

In [5]:
cols_binarias = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
    'StreamingTV', 'StreamingMovies', 'PaperlessBilling',
]
encoders_binarios = {}
for col in cols_binarias:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders_binarios[col] = le

cols_ohe = ['InternetService', 'Contract', 'PaymentMethod', 'tenure_grupo']
df = pd.get_dummies(df, columns=cols_ohe, drop_first=False)

bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

print(f'\nShape após encoding: {df.shape[0]:,} linhas x {df.shape[1]} colunas')


Shape após encoding: 7,043 linhas x 59 colunas


## 5. Seleção de features

Flags redundantes com OHE são removidas; interações do EDA são mantidas para maximizar F1 e acurácia.

In [6]:
colunas_excluir = [
    'customerID', 'Churn',
    'MonthlyCharges_orig', 'TotalCharges_orig', 'tenure_orig',
]
# Redundantes após OHE (duplicam Contract_*, InternetService_*, PaymentMethod_*, SeniorCitizen etc.)
colunas_redundantes = [
    'is_contrato_mensal', 'is_fibra_optica', 'is_boleto_eletronico', 'is_senior',
    'is_sem_suporte', 'is_sem_seguranca_online', 'score_risco_base',
]
feature_cols = [
    c for c in df.columns
    if c not in colunas_excluir and c not in colunas_redundantes
]

X = df[feature_cols]
y = df['Churn']

print(f'\nFeatures para modelagem: {len(feature_cols)}')
print(feature_cols)


Features para modelagem: 47
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'is_novo_cliente', 'charges_por_mes', 'charges_desvio_contrato', 'qtd_servicos', 'sem_servicos_adicionais', 'fibra_sem_suporte', 'qtd_servicos_protecao', 'sem_protecao', 'score_risco_estendido', 'contrato_mensal_novo', 'contrato_mensal_fibra', 'tenure_x_contrato_mensal', 'charges_x_contrato_mensal', 'boleto_x_contrato_mensal', 'fibra_x_sem_protecao', 'tenure_curto_fibra', 'alta_cobranca_relativa', 'InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No', 'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Bank transfer (automatic)', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check', 'tenure_grupo_0-12m', 't

## 6. Divisão estratificada e normalização

O escalonador é ajustado somente com os dados de treino.

In [7]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'\nTreino: {X_treino.shape[0]:,} amostras')
print(f"  Churn 0: {(y_treino == 0).sum():,} ({(y_treino == 0).mean() * 100:.1f}%)")
print(f"  Churn 1: {(y_treino == 1).sum():,} ({(y_treino == 1).mean() * 100:.1f}%)")
print(f'\nTeste: {X_teste.shape[0]:,} amostras')
print(f"  Churn 0: {(y_teste == 0).sum():,} ({(y_teste == 0).mean() * 100:.1f}%)")
print(f"  Churn 1: {(y_teste == 1).sum():,} ({(y_teste == 1).mean() * 100:.1f}%)")

assert abs((y_treino == 1).mean() - (y_teste == 1).mean()) < 0.01, (
    'ALERTA: Proporção de churn diferente entre treino e teste!'
)
print('\nValidação de estratificação: OK')

cols_normalizar = [
    'tenure', 'MonthlyCharges', 'TotalCharges',
    'charges_por_mes', 'charges_desvio_contrato', 'qtd_servicos',
    'qtd_servicos_protecao', 'score_risco_estendido',
    'tenure_x_contrato_mensal', 'charges_x_contrato_mensal',
]
cols_normalizar = [c for c in cols_normalizar if c in X_treino.columns]

X_treino = X_treino.copy()
X_teste = X_teste.copy()
X_treino[cols_normalizar] = X_treino[cols_normalizar].astype(float)
X_teste[cols_normalizar] = X_teste[cols_normalizar].astype(float)

scaler = StandardScaler()
X_treino[cols_normalizar] = scaler.fit_transform(X_treino[cols_normalizar])
X_teste[cols_normalizar] = scaler.transform(X_teste[cols_normalizar])


Treino: 5,634 amostras
  Churn 0: 4,139 (73.5%)
  Churn 1: 1,495 (26.5%)

Teste: 1,409 amostras
  Churn 0: 1,035 (73.5%)
  Churn 1: 374 (26.5%)

Validação de estratificação: OK


## 7. Exportação dos artefatos

In [8]:
Path('../models').mkdir(parents=True, exist_ok=True)
Path('../data/processed').mkdir(parents=True, exist_ok=True)

joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(encoders_binarios, '../models/encoders_binarios.pkl')
joblib.dump(feature_cols, '../models/feature_cols.pkl')
print('\nScaler e encoders salvos em ../models/')

treino = X_treino.copy()
treino['Churn'] = y_treino.values
treino['MonthlyCharges_orig'] = df.loc[X_treino.index, 'MonthlyCharges_orig'].values
treino['TotalCharges_orig'] = df.loc[X_treino.index, 'TotalCharges_orig'].values
treino['tenure_orig'] = df.loc[X_treino.index, 'tenure_orig'].values

teste = X_teste.copy()
teste['Churn'] = y_teste.values
teste['MonthlyCharges_orig'] = df.loc[X_teste.index, 'MonthlyCharges_orig'].values
teste['TotalCharges_orig'] = df.loc[X_teste.index, 'TotalCharges_orig'].values
teste['tenure_orig'] = df.loc[X_teste.index, 'tenure_orig'].values

treino.to_parquet('../data/processed/treino_modelagem.parquet', index=False)
teste.to_parquet('../data/processed/teste_modelagem.parquet', index=False)

print('\nConjuntos exportados:')
print(f'  treino_modelagem.parquet: {treino.shape[0]:,} linhas x {treino.shape[1]} colunas')
print(f'  teste_modelagem.parquet:  {teste.shape[0]:,} linhas x {teste.shape[1]} colunas')
print('\nFeature Store (Notebook 03) processado com sucesso!')


Scaler e encoders salvos em ../models/

Conjuntos exportados:
  treino_modelagem.parquet: 5,634 linhas x 51 colunas
  teste_modelagem.parquet:  1,409 linhas x 51 colunas

Feature Store (Notebook 03) processado com sucesso!
